In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
batch_size = 100
# batch_size = None

# opt = torch_numopt.GaussNewton(model=model, lr=1, line_search_method="const")
opt = torch_numopt.GaussNewton(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", batch_size=batch_size)
# opt = torch_numopt.GaussNewton(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method='backtrack', line_search_cond='wolfe')
# opt = torch_numopt.GaussNewton(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method='backtrack', line_search_cond='strong-wolfe')
# opt = torch_numopt.GaussNewton(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method='backtrack', line_search_cond='goldstein')

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.14343106746673584
epoch:  1, loss: 0.03985626995563507
epoch:  2, loss: 0.03865312784910202
epoch:  3, loss: 0.037443891167640686
epoch:  4, loss: 0.036050260066986084
epoch:  5, loss: 0.034561190754175186
epoch:  6, loss: 0.03319666534662247
epoch:  7, loss: 0.03174341097474098
epoch:  8, loss: 0.030395707115530968
epoch:  9, loss: 0.02906080149114132
epoch:  10, loss: 0.027800757437944412
epoch:  11, loss: 0.026618285104632378
epoch:  12, loss: 0.02545047737658024
epoch:  13, loss: 0.024305814877152443
epoch:  14, loss: 0.023139862343668938
epoch:  15, loss: 0.022003192454576492
epoch:  16, loss: 0.02090957574546337
epoch:  17, loss: 0.019939590245485306
epoch:  18, loss: 0.018913326784968376
epoch:  19, loss: 0.01782379113137722
epoch:  20, loss: 0.016844358295202255
epoch:  21, loss: 0.015965955331921577
epoch:  22, loss: 0.015229826793074608
epoch:  23, loss: 0.014488058164715767
epoch:  24, loss: 0.013857749290764332
epoch:  25, loss: 0.013240167871117592
epoch

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.843895435333252
Test metrics:  R2 = 0.6328291893005371
